# Lesson 5.1 — OCI Registries & Chart Publishing

**Phase 5: Real-World Workflows**

---

## Learning Objectives

By the end of this lesson you will be able to:
- Package a chart with `helm package`
- Push charts to an OCI registry with `helm push`
- Pull and install charts directly from an OCI registry with `oci://`
- Understand the difference between OCI registries and traditional chart repos

---

## OCI vs Traditional Helm Repos

| | Traditional Repo | OCI Registry |
|---|---|---|
| Protocol | HTTP with `index.yaml` | OCI (same as Docker images) |
| Add repo? | `helm repo add` required | No — use `oci://` prefix directly |
| Examples | Bitnami, Artifact Hub | GHCR, ECR, ACR, Docker Hub, Harbor |
| Auth | Repo-specific | `helm registry login` (like `docker login`) |
| Helm version | All versions | Helm 3.8+ (GA) |

> **Reference:** [Use OCI-based Registries](https://helm.sh/docs/topics/registries/) — https://helm.sh/docs/topics/registries/

---

## Step 1: Package Your Chart

In [ ]:
# Create a sample chart to publish
rm -rf /tmp/mypublishedchart
helm create /tmp/mypublishedchart

# Update version to be explicit
sed -i 's/version: 0.1.0/version: 1.0.0/' /tmp/mypublishedchart/Chart.yaml
cat /tmp/mypublishedchart/Chart.yaml | grep -E '^(name|version|appVersion)'

In [ ]:
# Package: creates a .tgz archive in the current directory (or --destination)
helm package /tmp/mypublishedchart --destination /tmp/packaged

ls -lh /tmp/packaged/

The resulting `.tgz` contains all chart files. The filename includes the version: `mypublishedchart-1.0.0.tgz`.

---

## Step 2: Authenticate with an OCI Registry

Authentication works like `docker login`:

In [ ]:
# Examples — replace with your registry details
echo "== OCI Registry Login Examples =="
echo ""
echo "# GitHub Container Registry"
echo 'echo $GITHUB_TOKEN | helm registry login ghcr.io -u <username> --password-stdin'
echo ""
echo "# AWS ECR (get token with AWS CLI)"
echo 'aws ecr get-login-password | helm registry login <account>.dkr.ecr.<region>.amazonaws.com -u AWS --password-stdin'
echo ""
echo "# Azure Container Registry"
echo 'az acr login --name <registry-name>'
echo ""
echo "# Docker Hub"
echo 'helm registry login registry-1.docker.io -u <username> --password-stdin'
echo ""
echo "# Harbor (self-hosted)"
echo 'helm registry login harbor.mycompany.com -u admin --password-stdin'

---

## Step 3: Push to an OCI Registry

In [ ]:
# The push command format:
# helm push <chart.tgz> oci://<registry>/<namespace>

echo "== Push examples =="
echo ""
echo "# Push to GitHub Container Registry"
echo 'helm push /tmp/packaged/mypublishedchart-1.0.0.tgz oci://ghcr.io/myorg/charts'
echo ""
echo "# Push to ECR"
echo 'helm push /tmp/packaged/mypublishedchart-1.0.0.tgz oci://<account>.dkr.ecr.<region>.amazonaws.com'
echo ""
echo "# After push, the chart is available at:"
echo 'oci://ghcr.io/myorg/charts/mypublishedchart:1.0.0'

---

## Step 4: Install from an OCI Registry

In [ ]:
# Install directly — no 'helm repo add' needed!
echo "== Install from OCI registry examples =="
echo ""
echo '# Install a specific version'
echo 'helm install my-app oci://ghcr.io/myorg/charts/mypublishedchart --version 1.0.0'
echo ""
echo '# Install with values file'
echo 'helm install my-app oci://ghcr.io/myorg/charts/mypublishedchart --version 1.0.0 -f production.yaml'
echo ""
echo '# Show chart metadata without installing'
echo 'helm show chart oci://ghcr.io/myorg/charts/mypublishedchart --version 1.0.0'

echo ""
echo "== Public chart example (no auth needed) =="
# Pull the public podinfo chart to inspect it
helm pull oci://ghcr.io/stefanprodan/charts/podinfo --version 6.11.2 --destination /tmp/ 2>/dev/null
ls /tmp/podinfo*.tgz 2>/dev/null && echo "Chart downloaded successfully" || echo "Download skipped (no network)"

---

## Verifying Chart Provenance

For production use, sign your charts with `helm package --sign` (requires GPG) and verify on install with `--verify`:

In [ ]:
echo "== Chart signing commands (requires GPG setup) =="
echo ""
echo '# Generate a signing key'
echo 'gpg --gen-key'
echo ""
echo '# Package and sign'
echo 'helm package --sign --key "Your Name" --keyring ~/.gnupg/secring.gpg /tmp/mypublishedchart'
echo '# This creates: mypublishedchart-1.0.0.tgz + mypublishedchart-1.0.0.tgz.prov'
echo ""
echo '# Install and verify'
echo 'helm install my-app /path/to/mypublishedchart-1.0.0.tgz --verify'

---

## Setting Up a Local OCI Registry for Testing

For local development without a cloud registry:

In [ ]:
echo "== Local OCI registry with Docker (requires Docker to be running) =="
echo ""
echo '# Start a local registry on port 5000'
echo 'docker run -d -p 5000:5000 --name helm-registry registry:2'
echo ""
echo '# Push to local registry (no auth needed)'
echo 'helm push /tmp/packaged/mypublishedchart-1.0.0.tgz oci://localhost:5000/charts'
echo ""
echo '# Install from local registry'
echo 'helm install my-app oci://localhost:5000/charts/mypublishedchart --version 1.0.0'
echo ""
echo '# Stop registry when done'
echo 'docker stop helm-registry && docker rm helm-registry'

---

## Exercise

In [ ]:
# TASK 1: Create a new chart called 'myservice' with version 0.5.0.
#         Package it to /tmp/myservice-packaged/
#         Verify the .tgz was created and contains the right files.
# TODO:


# TASK 2: Use 'helm show chart' to inspect the podinfo chart from
#         oci://ghcr.io/stefanprodan/charts/podinfo --version 6.11.2
#         Without downloading or installing it.
# TODO:
# helm show chart oci://ghcr.io/stefanprodan/charts/podinfo --version 6.11.2


# TASK 3: If you have Docker available, start a local registry,
#         push your myservice chart to it, and then install from it.
# TODO (bonus):

echo "Exercise: implement the tasks above"

---

## Summary

- `helm package <chart>` creates a `.tgz` archive versioned by `Chart.yaml`.
- `helm registry login <registry>` authenticates (like `docker login`).
- `helm push <chart.tgz> oci://<registry>/<namespace>` publishes to an OCI registry.
- `helm install <name> oci://<registry>/<chart> --version <ver>` installs without `helm repo add`.
- OCI registries (GHCR, ECR, ACR) are the modern recommended distribution method.

---

## Next Lesson

➡️ [Lesson 5.2 — Helm Test](lesson-5.2-helm-test.ipynb)